# 14 - A hosted agent with user-delegated SharePoint knowledge

## Required permissions

- **Setup author:** **Foundry Owner on the parent Foundry resource**, **Search Service Contributor** + **Search Index Data Reader** on Search, `az login` in the deployment tenant. Creating the project connection requires `Microsoft.CognitiveServices/accounts/projects/connections/write`.
- **Project access to Search:** the **Foundry project's managed identity** must already have **Search Index Data Reader** on your **Azure AI Search service**. The [deployment script](../scripts/deploy.ps1) provisions and verifies this assignment ([Bicep definition](../infra/data.bicep)). For an existing environment not deployed with these scripts, ask an administrator to configure it before starting. This notebook does not assign roles. This service permission lets the Toolbox call Search; users' SharePoint permissions still determine which documents they can retrieve.
- **Question callers:** signed-in, non-administrator users need Foundry endpoint invocation access and appropriate SharePoint document access. For this Foundry IQ source-authorization path, the user's Search token does not require a direct Search service role; the project managed identity supplies outer service authentication. Lab 07's separate direct-query prerequisites still apply there. Setup rights are not a substitute for user authorization. Use separate ordinary Alice/Bob accounts for the exercise.
- **Existing services:** retain lab 07's ingestion/ACL setup and Search's model/vectorizer permissions from lab 11. This notebook creates no app registration and changes no indexing, SharePoint permissions or consent grants.

**Resource prefix:** reuse lab 07/11's `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). Reuse `alice01-kb-sharepoint` -> `alice01-nb7-index`; create/update `alice01-toolbox-kb`, `alice01-iq-search-mcp`, `alice01-iq-toolbox`, `alice01-iq-agent` and the local `alice01-hosted-agent-iq` folder.

**Intentional migration:** rerunning with an existing lab 14 prefix repoints its knowledge base and connection from the old scenario to ACL-aware SharePoint. An independent prefix is not a shortcut: prepare labs 07 and 11 for that prefix first. Names avoid collisions, not enforce access isolation.

Notebook 13 used a Python function tool. Here native `FoundryToolbox` connects the hosted agent to a Foundry IQ knowledge base, preserving each incoming user's platform context.

You will validate the existing source/index without writing them, create a SharePoint-only knowledge base and toolbox connection, generate and deploy a hosted agent, and compare the same sensitive question as two different users.

Set the kernel working directory to `python-notebooks`. Configuration loads `..\.env` first, then `..\.env.azure`, both with `override=True`; set `WORKSHOP_PREFIX` before running (no prefix prompt). Setup, deployment and questions are live/billable operations when you run them; offline tests do not contact Azure.

## Architecture: service authentication plus user source authorization

```text
Signed-in USER -> Foundry audience / AzureRbac agent endpoint
  -> ResponsesHostServer (Responses protocol 2.0.0)
  -> per-run user_id + call_id guard -> agent model
  -> native FoundryToolbox (forwards the current platform call ID)
  -> project connection (ProjectManagedIdentity, Search audience)
  -> Foundry IQ MCP -> <prefix>-toolbox-kb
  -> ONLY <prefix>-kb-sharepoint -> <prefix>-nb7-index
  -> indexed SharePoint user/group ACLs -> permitted evidence + citations
```

Agent Service dynamically supplies `x-ms-query-source-authorization` with the signed-in user's Search token. The notebook and container **never** construct that header, place user tokens in prompts or shared toolbox headers, or forward arbitrary user-identity headers. Native integration resolves the current call context separately for each question.

| Identity / component | Responsibility |
|---|---|
| Setup author's CLI user | Author the lab objects and deploy source; not the identity used to prove Alice/Bob isolation. |
| Incoming question's CLI user | Authenticate to the normal Foundry audience, `https://ai.azure.com`; Foundry obtains the user's downstream Search context for source authorization. |
| Project managed identity | Authenticate the project connection to Search using **Search Index Data Reader**. This service role does not grant Alice or Bob access to SharePoint facts. |
| Hosted agent identity | Authenticate model/toolbox access. It is not the Search connection's service principal in this scenario. |
| Search service identity | Retain existing model/vectorizer permissions; lab 07's ingestion pipeline has already projected document ACLs. |
| Project connection and toolbox | Target and versioned tool configuration; `ProjectManagedIdentity`, `RemoteTool`, Search audience, `ApiType: Azure`. No new app or secret. |

Tool approval (`require_approval='never'`) is not authorization. Inspect tool evidence and source citations: fluent text is not proof of retrieval or ACL enforcement. There are two billable model layers (agent and knowledge-base planning/synthesis), plus hosted compute and Search.

The previous rendered architecture image describes a different identity flow and is intentionally not used here. This text flow describes the delegated scenario.

## Prerequisites

- Complete **07** and **11** with cleanup disabled, using this prefix and Search service: `<prefix>-kb-sharepoint` must be a `searchIndex` source referencing **exactly** `<prefix>-nb7-index`; `<prefix>-kb` supplies model settings.
- The index must have `permissionFilterOption: enabled`, non-retrievable/filterable `UserIds` and `GroupIds` permission fields, `semantic-config`, and retrievable content/source metadata. Section 1 fails closed on a mismatch. Existing indexed ACLs must be current; schema checks alone cannot prove ACL correctness.
- Configure `AZURE_AI_PROJECT_RESOURCE_ID`; the project must already have a managed identity. This notebook will not enable an identity implicitly.
- Use a logged-in **user**, not a service principal, managed identity or app-only CLI login, for questions. Notebook 13 must already work in your project.
- Prepare ordinary Alice and Bob accounts and a safe test document containing a distinctive sensitive fact: Alice can open it in SharePoint, Bob cannot. Confirm the source permissions directly, and allow ingestion/ACL propagation before testing.

Only the existing indexed SharePoint source is selected. No Blob or web sources, new app registrations, uploads, index writes or indexer runs are part of this lab.

In [ ]:
# Run with the kernel working directory set to python-notebooks.
import base64
import json
import os
import re
import time
import zipfile
from pathlib import Path
from uuid import UUID

import requests
from azure.ai.projects import AIProjectClient
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path('..')
load_dotenv(r'..\.env', override=True)
load_dotenv(r'..\.env.azure', override=True)
required = ('WORKSHOP_PREFIX', 
    'AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'AZURE_SEARCH_RESOURCE_ID',
    'AZURE_AI_PROJECT_ENDPOINT', 'AZURE_AI_PROJECT_RESOURCE_ID', 'AZURE_AI_MODEL_DEPLOYMENT_NAME',
)
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
search_resource_id = os.environ['AZURE_SEARCH_RESOURCE_ID']
project_endpoint = os.environ['AZURE_AI_PROJECT_ENDPOINT'].rstrip('/')
project_resource_id = os.environ['AZURE_AI_PROJECT_RESOURCE_ID'].rstrip('/')
model_name = os.environ['AZURE_AI_MODEL_DEPLOYMENT_NAME']
prefix = os.environ['WORKSHOP_PREFIX'].strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

api_version = '2026-08-01-preview'
sharepoint_source_name = f'{prefix}-kb-sharepoint'
sharepoint_index_name = f'{prefix}-nb7-index'
knowledge_base_name = f'{prefix}-toolbox-kb'
toolbox_name = f'{prefix}-iq-toolbox'
connection_name = f'{prefix}-iq-search-mcp'
connection_id = f'{project_resource_id}/connections/{connection_name}'
agent_name = f'{prefix}-iq-agent'
agent_folder = root / f'{prefix}-hosted-agent-iq'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
arm_token = get_bearer_token_provider(credential, 'https://management.azure.com/.default')
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential, allow_preview=True)
print('Knowledge base:', knowledge_base_name)
print('Toolbox:', toolbox_name, '| connection:', connection_name)
print('Agent:', agent_name)

## 1. Validate the existing SharePoint source and create the knowledge base

Read lab 11's source, then lab 07's exact index and verify its permission schema before any knowledge-base PUT. Copy only the model settings and answer instructions from `<prefix>-kb`; select exactly one knowledge source. These direct Search REST calls use your setup token; they do not retrieve documents or prove runtime user delegation.

In [ ]:
response = requests.get(
    f'{search_endpoint}/knowledgesources/{sharepoint_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{sharepoint_source_name} not found. Complete notebooks 07 and 11 first (CLEAN_UP = False).')
response.raise_for_status()
sharepoint_source = response.json()
source_parameters = sharepoint_source.get('searchIndexParameters', {})
if (sharepoint_source.get('kind') != 'searchIndex'
        or source_parameters.get('searchIndexName') != sharepoint_index_name):
    raise RuntimeError(f'{sharepoint_source_name} must be a searchIndex source for exactly {sharepoint_index_name}.')
if source_parameters.get('semanticConfigurationName') != 'semantic-config':
    raise RuntimeError('The SharePoint source must use semantic-config from notebook 07.')
required_source_fields = {'id', 'content', 'sourceFile', 'webUrl', 'page', 'pageTo'}
if (not required_source_fields.issubset({field.get('name') for field in source_parameters.get('sourceDataFields', [])})
        or 'content' not in {field.get('name') for field in source_parameters.get('searchFields', [])}):
    raise RuntimeError('The SharePoint source must expose lab 07 content and source metadata. Repair it in notebook 11.')

response = requests.get(
    f'{search_endpoint}/indexes/{sharepoint_index_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
response.raise_for_status()
sharepoint_index = response.json()
if sharepoint_index.get('permissionFilterOption') != 'enabled':
    raise RuntimeError('SharePoint index permissionFilterOption must be enabled. Stop and repair notebook 07 setup.')
fields = {field['name']: field for field in sharepoint_index.get('fields', [])}
for field_name, permission_kind in (('UserIds', 'userIds'), ('GroupIds', 'groupIds')):
    field = fields.get(field_name, {})
    if (field.get('type') != 'Collection(Edm.String)'
            or field.get('permissionFilter') != permission_kind
            or field.get('filterable') is not True or field.get('retrievable') is not False):
        raise RuntimeError(f'Invalid {field_name} permission field. Stop and repair notebook 07 setup.')
for field_name in required_source_fields:
    if field_name not in fields or fields[field_name].get('retrievable', True) is not True:
        raise RuntimeError(f'SharePoint index must expose retrievable {field_name}.')
semantic_configs = sharepoint_index.get('semantic', {}).get('configurations', [])
semantic_config = next((config for config in semantic_configs if config.get('name') == 'semantic-config'), None)
if (not semantic_config
        or 'content' not in {field.get('fieldName') for field in semantic_config.get(
            'prioritizedFields', {}).get('prioritizedContentFields', [])}):
    raise RuntimeError('SharePoint index needs semantic-config with prioritized content from notebook 07.')

response = requests.get(
    f'{search_endpoint}/knowledgebases/{prefix}-kb?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{prefix}-kb not found. Complete notebook 11 first (CLEAN_UP = False).')
response.raise_for_status()
base_kb = response.json()

knowledge_base = {
    'name': knowledge_base_name,
    'description': 'ACL-aware indexed SharePoint documents for the hosted agent.',
    'retrievalInstructions': f'Use {sharepoint_source_name} for questions about company policy, travel, onboarding or documents.',
    'answerInstructions': base_kb.get('answerInstructions'),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': sharepoint_source_name}],
    'models': base_kb['models'],
    'retrievalReasoningEffort': {'kind': 'medium'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()

knowledge_base_mcp_url = f'{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version={api_version}'
print('Knowledge base ready:', knowledge_base_name)
print('MCP endpoint:', knowledge_base_mcp_url)

## 2. Create the connection and toolbox

The visible ARM PUT creates/updates `{prefix}-iq-search-mcp` with **ProjectManagedIdentity** service authentication, category `RemoteTool`, audience `https://search.azure.com` and metadata `ApiType: Azure`, targeting the knowledge base MCP URL. No key is stored. This intentionally repoints a previous lab 14 connection using the same prefix; old shared connections are untouched.

Foundry IQ uses **target-specific source authorization**, not a generic `UserEntraToken` connection. Agent Service obtains the signed-in user's Search context in addition to the project's service authentication. Keep the normal AzureRbac / Foundry audience endpoint; no AgentBlueprint setup is needed.

The toolbox's MCP tool references `server_url` and `project_connection_id`; `require_approval='never'` removes approval prompts, not ACL checks. Each toolbox version is immutable; the agent follows the default version. Publishing/promoting a compatible version can change tools without redeploying the agent.

In [ ]:
from azure.ai.projects.models import MCPToolboxTool

connection_response = requests.put(
    f'https://management.azure.com{connection_id}?api-version=2025-04-01-preview',
    headers={'Authorization': f'Bearer {arm_token()}'},
    json={'properties': {
        'category': 'RemoteTool',
        'authType': 'ProjectManagedIdentity',
        'target': knowledge_base_mcp_url,
        'audience': 'https://search.azure.com',
        'isSharedToAll': True,
        'metadata': {'ApiType': 'Azure'},
    }},
    timeout=(10, 90),
)
connection_response.raise_for_status()
print('Connection ready:', connection_name)

toolbox_version = project_client.toolboxes.create_version(
    name=toolbox_name,
    description='Foundry IQ knowledge base for the workshop hosted agent.',
    tools=[
        MCPToolboxTool(
            server_label='knowledge_base',
            server_url=knowledge_base_mcp_url,
            require_approval='never',
            project_connection_id=connection_id,
        ),
    ],
)
toolbox = project_client.toolboxes.get(toolbox_name)
if toolbox.default_version != toolbox_version.version:
    toolbox = project_client.toolboxes.update(toolbox_name, default_version=toolbox_version.version)
print('Toolbox', toolbox.name, '- new version', toolbox_version.version, '- default version', toolbox.default_version)

## 3. Write the agent

Native `FoundryToolbox(credential)` discovers tools using `TOOLBOX_NAME` and forwards the platform's **per-request call ID**. `ResponsesHostServer` supplies request context. The agent middleware below requires both `get_request_context().user_id` and `.call_id` **at agent run time**, before model/tool calls. Startup/readiness tool discovery has no user and is deliberately not blocked.

This is a fail-closed run guard, **not a standalone authentication boundary**: Foundry must validate the incoming user and platform context. Never expose an unauthenticated container as an equivalent endpoint or synthesize caller headers. The source contains no user tokens, Search URL, key, or knowledge-base name. No session or conversation is shared between users.

In [ ]:
main_py = """import os

from agent_framework import Agent, AgentMiddleware
from agent_framework.foundry import FoundryChatClient
from agent_framework_foundry_hosting import FoundryToolbox, ResponsesHostServer
from azure.ai.agentserver.core import get_request_context
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

# Finds the toolbox from FOUNDRY_PROJECT_ENDPOINT (set by the platform) and TOOLBOX_NAME.
toolbox = FoundryToolbox(credential)

client = FoundryChatClient(
    project_endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=credential,
    timeout=120.0,
    function_invocation_configuration={
        'max_iterations': 3, 'max_function_calls': 2, 'max_consecutive_errors_per_request': 1,
    },
)
client.client.max_retries = 0

class RequireDelegatedContext(AgentMiddleware):
    async def process(self, context, call_next):
        request_context = get_request_context()
        if (request_context is None
                or not request_context.user_id or not request_context.call_id):
            raise PermissionError("Signed-in user and platform call context are required; stop without retrieval.")
        await call_next()


agent = Agent(
    client=client,
    instructions=(
        "For document questions, always use the knowledge base tool for this request. "
        "Answer only from evidence returned for the current signed-in user. "
        "If a tool fails authorization or requests consent, stop and explain that retrieval is incomplete. "
        "If the knowledge base does not contain the answer, say you do not know. "
        "Cite the documents you used."
    ),
    tools=toolbox,
    middleware=[RequireDelegatedContext()],
    default_options={"store": False},
)

if __name__ == "__main__":
    ResponsesHostServer(agent).run()
"""

requirements_txt = """agent-framework-foundry
agent-framework-foundry-hosting>=1.0.0b261112
"""

agent_folder.mkdir(exist_ok=True)
(agent_folder / 'main.py').write_text(main_py, encoding='utf-8')
(agent_folder / 'requirements.txt').write_text(requirements_txt, encoding='utf-8')
compile(main_py, 'main.py', 'exec')
print('Wrote', [p.name for p in agent_folder.iterdir()], 'to', agent_folder)

## 4. Deploy the agent

Same deployment as notebook 13. The only difference is the `TOOLBOX_NAME` environment variable that points the agent at your toolbox. The cell then polls until the build is active (up to 15 minutes).

In [ ]:
from azure.ai.projects.models import CodeConfiguration, HostedAgentDefinition, ProtocolVersionRecord

zip_path = agent_folder / 'agent.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(agent_folder / 'main.py', 'main.py')
    bundle.write(agent_folder / 'requirements.txt', 'requirements.txt')

definition = HostedAgentDefinition(
    cpu='0.5',
    memory='1Gi',
    protocol_versions=[ProtocolVersionRecord(protocol='responses', version='2.0.0')],
    environment_variables={'AZURE_AI_MODEL_DEPLOYMENT_NAME': model_name, 'TOOLBOX_NAME': toolbox_name},
    code_configuration=CodeConfiguration(
        runtime='python_3_13',
        entry_point=['python', 'main.py'],
        dependency_resolution='remote_build',
    ),
)

with open(zip_path, 'rb') as code_stream:
    version = project_client.agents.create_version_from_code(
        agent_name,
        definition=definition,
        code=code_stream,
        description='Hosted agent grounded in a Foundry IQ knowledge base through a toolbox',
        metadata={'workshop': 'iq', 'attendee': prefix},
    )
print('Created version', version.version)

deadline = time.time() + 15 * 60
while True:
    version = project_client.agents.get_version(agent_name, version.version)
    status = str(getattr(version.status, 'value', version.status)).lower()
    print('status:', status)
    if status == 'active':
        break
    if status == 'failed':
        raise RuntimeError(f'Build failed: {version}')
    if time.time() > deadline:
        raise TimeoutError('Still not active after 15 minutes.')
    time.sleep(15)
print('Agent version is active:', agent_name, 'version', version.version)

## 5. Ask as a signed-in USER

Every independent question creates a **fresh AzureCliCredential**, so switching `az login` cannot silently reuse the previous user's credential object. It checks the Foundry token locally before the question network call: delegated `scp`, user `oid`, expected tenant and audience, expiry, and not `idtyp: app`. JWT decoding here is only a local sanity check, **not signature verification**; Foundry validates the real token. Tokens are never printed or placed in prompts.

The endpoint stays on normal **AzureRbac** with the Foundry audience, not AgentBlueprint. Each call is fresh (`store=False`, no previous response or conversation/session ID), bounded by a timeout with SDK retries disabled. The exact checked Foundry token authenticates that one request. No downstream token is handled by this client.

On missing user context, 401/403, consent or tool errors, **stop**, investigate the intended user's permissions/context and source setup; do not fall back to app-only access or automatically retry. If `oauth_consent_request` appears, no completed retrieval is claimed: use the product-owned consent experience as that user (or ask your tenant administrator if policy blocks it), then begin a new deliberate request. This teaching client does not automate consent/resumption or display untrusted sign-in URLs.

In [ ]:
def validate_user_token(token, expected_tenant):
    # Local sanity check only; Foundry verifies the token's signature and authority.
    try:
        parts = token.split('.')
        if len(parts) != 3:
            raise ValueError()
        payload = parts[1] + '=' * (-len(parts[1]) % 4)
        claims = json.loads(base64.urlsafe_b64decode(payload))
        if (not isinstance(claims, dict) or claims.get('idtyp') == 'app'
                or not isinstance(claims.get('scp'), str) or not claims['scp'].strip()
                or not claims.get('oid')):
            raise ValueError()
        UUID(claims['oid'])
        if str(UUID(claims['tid'])) != str(UUID(expected_tenant)):
            raise ValueError()
        if claims.get('aud') not in ('https://ai.azure.com', 'https://ai.azure.com/'):
            raise ValueError()
        expires = claims.get('exp')
        if isinstance(expires, bool) or not isinstance(expires, int) or expires <= time.time():
            raise ValueError()
    except (ValueError, TypeError, KeyError, AttributeError, UnicodeError):
        raise PermissionError(
            'A current delegated USER token for this tenant and https://ai.azure.com is required. '
            'Stop; sign in with az login as the intended user. No app-only fallback.'
        ) from None
    return claims


def response_nodes(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from response_nodes(child)
    elif isinstance(value, list):
        for child in value:
            yield from response_nodes(child)
    elif isinstance(value, str):
        try:
            decoded = json.loads(value)
        except ValueError:
            return
        if isinstance(decoded, (dict, list)):
            yield from response_nodes(decoded)


def ask_as_current_user(question):
    with AzureCliCredential(tenant_id=tenant_id) as user_credential:
        access_token = user_credential.get_token('https://ai.azure.com/.default')
        claims = validate_user_token(access_token.token, tenant_id)
        print('Question user object ID:', claims['oid'], '| tenant:', claims['tid'])
        with AIProjectClient(endpoint=project_endpoint, credential=user_credential, allow_preview=True) as user_project_client:
            with user_project_client.get_openai_client(
                agent_name=agent_name, api_key=access_token.token, max_retries=0, timeout=120.0,
            ) as user_openai_client:
                try:
                    result = user_openai_client.responses.create(input=question, store=False)
                except Exception:
                    raise RuntimeError(
                        'Question failed. Stop; inspect user authentication, consent, source access and service roles. '
                        'No automatic retry or application-identity fallback.'
                    ) from None
    body = result.model_dump(mode='json')
    if 'oauth_consent_request' in json.dumps(body):
        raise PermissionError(
            'oauth_consent_request: retrieval is not complete. Stop and use the product-owned user consent '
            'experience; ask your tenant administrator if policy blocks consent. No automatic retry.'
        )
    nodes = list(response_nodes(body))
    if result.status != 'completed' or any(
        node.get('error') or node.get('is_error') or node.get('isError')
        or node.get('status') in ('failed', 'incomplete', 'cancelled')
        for node in nodes
    ):
        raise RuntimeError('Retrieval/response did not complete successfully. Stop; inspect tool/authentication evidence.')
    print(result.output_text)
    for item in result.output:
        print('output item:', item.type, getattr(item, 'name', '') or '')
    citations = [node for node in nodes if node.get('type') in ('url_citation', 'file_citation')]
    for citation in citations:
        print('source citation:', citation.get('title', ''), citation.get('url') or citation.get('file_id', ''))
    print('Check tool evidence and source citations against SharePoint; text alone is not proof of ACL enforcement.')
    return result


response = ask_as_current_user('What is our policy for booking international travel?')

## 6. Check grounding, then compare Alice and Bob

An unrelated question should get "I don't know" instead of an invented document answer. The next cell makes another independent user invocation; it does not reuse response history.

**Required two-user live exercise (manual, not execute-all):**

1. Prepare a safe restricted test fact in the indexed SharePoint library. Verify **directly in SharePoint** that ordinary Alice can open the document and ordinary Bob cannot. Do not use setup, administrator or elevated accounts as either test user.
2. Choose **one exact sensitive fact question**, without supplying the secret fact in the question. Use that identical question for both users and the **same deployed agent/toolbox/KB**. Record the question and expected document URL, not secret content in a committed notebook.
3. In a separate fresh conversation/kernel, sign in as Alice using Azure CLI. Configure the notebook and define the invocation helpers without rerunning the authoring/deployment cells. The displayed user object ID must match Alice. Deliberately run `ask_as_current_user(sensitive_question)` once.
4. Repeat in another fresh conversation/kernel after `az login` as Bob; confirm Bob's different object ID. Run **exactly** `ask_as_current_user(sensitive_question)` with no previous response/session and no copied Alice answer or tool result.
5. Alice should receive the permitted fact with a genuine knowledge-base tool call and citation to that document. Bob must receive **neither that fact nor a restricted citation/snippet** (an honest no-evidence answer is expected). Inspect tool results/traces and citations too, not just the final wording; if evidence is not exposed, record that limitation rather than claiming success.
6. Missing user/call context, auth failures or consent requests are **failed/incomplete verification**, not an ACL-denied success. Stop and correct the intended user's setup; never retry as a privileged identity or app. A leak is a stop condition: inspect the source ACL projections/sync and platform delegation, not prompt wording.

Index permission synchronization has latency. An offline fake-token test validates client guards and payloads, **not live ACL enforcement**, group membership resolution or tenant/platform consent. Do not declare the exercise passed until deployed end-to-end results match the source access.

In [ ]:
response = ask_as_current_user('What is the capital of Australia?')

## 7. Why the project role does not grant every user the same documents

There are **two separate authorization inputs** at Search. The connection's `ProjectManagedIdentity` authenticates the project service to the KB MCP endpoint. That project principal needs Search Index Data Reader; assigning it to the agent principal would not satisfy this connection. The role assignment is service access, not an instruction to bypass document permissions.

For Foundry IQ, Agent Service also resolves the **current signed-in user** from the incoming agent request and dynamically sets `x-ms-query-source-authorization` with that user's Search token. This target-specific behavior applies to Foundry-audience requests; it is not implemented by changing this connection to generic `UserEntraToken` (Search is not in that generic tools support table). This lab keeps AzureRbac and the Foundry audience rather than configuring an AgentBlueprint.

`ResponsesHostServer` protocol 2.0.0 exposes platform user/call context. The per-run middleware fails before inference if either is missing. Native `FoundryToolbox` forwards the **current** call ID so Agent Service can resolve the matching user; it must not be cached from Alice for Bob. Initial tool discovery can happen without a user, so the guard applies to question runs, not startup. The container does not verify bearer signatures itself: its guard depends on Foundry's authenticated platform boundary.

The existing lab 07 index has `permissionFilterOption: enabled` and `UserIds`/`GroupIds` permission filters populated by SharePoint ingestion. Search applies source authorization to the indexed content, and returns only eligible evidence for synthesis. The exact source/index/schema checks prevent accidentally selecting a non-ACL source, but do not validate the indexed ACL values or current SharePoint/group membership; test those end to end.

No client or generated source manually copies a Search user token, sends caller IDs as improvised headers, creates an app registration, grants consent, changes SharePoint ACLs or broadens sources. `isSharedToAll` shares the project connection configuration; it is **not** a SharePoint document grant. A missing user context/consent failure cannot be repaired by substituting project or app credentials.

Official references:

- [Connect agents to Foundry IQ (connection contract)](https://learn.microsoft.com/azure/foundry/agents/how-to/foundry-iq-connect)
- [Foundry IQ source authorization](https://learn.microsoft.com/azure/foundry/agents/concepts/user-delegation#foundry-iq-source-authorization)
- [Use a toolbox with a hosted agent (native caller context)](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/use-toolbox-hosted-agent)
- [Toolbox authentication](https://learn.microsoft.com/azure/foundry/agents/how-to/tools/tool-authentication)

## Cleanup

Opt in with `CLEAN_UP = True` to delete **only four lab 14-owned objects**: your agent, toolbox, knowledge base and prefixed project connection. Lab 07/11 sources, indexes, base KB, SharePoint documents, role assignments and generated local files remain. This migration does not broaden cleanup to remove any of them or an old shared connection.

In [ ]:
CLEAN_UP = False

if CLEAN_UP:
    project_client.agents.delete(agent_name)
    project_client.toolboxes.delete(toolbox_name)
    response = requests.delete(
        f'https://management.azure.com{connection_id}?api-version=2025-04-01-preview',
        headers={'Authorization': f'Bearer {arm_token()}'}, timeout=(10, 60),
    )
    if response.status_code != 404:
        response.raise_for_status()
    response = requests.delete(
        f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 60),
    )
    if response.status_code != 404:
        response.raise_for_status()
    print('Deleted', agent_name, toolbox_name, knowledge_base_name, connection_name)
else:
    print('Cleanup skipped. Set CLEAN_UP = True to delete', agent_name, toolbox_name, knowledge_base_name, connection_name)